In [1]:
!pip install stable-baselines3[extra] ale_py autorom[accept-rom-license]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 25.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 82.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 12.3 MB/s eta 0:00:00
  Created wheel for AutoROM.accept-rom-license: filename=autorom_accept_rom_license-0.6.1-py3-none-any.whl size=446710 sha256=8f7662cf0b763ec7beb218475faf84c5ffca1110cba43f4865e0213306dd3354
  Stored in directory: /root/.cache/pip/wheels/99/f1/ff/c6966c034a8259164bdc9deb4d1ea839f119474638100e6645
Successfully built AutoROM.accept-rom-license
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.3.0
    Uninstalling gymnasium-1.3.0:
      Successfully uninstalled gymnasium-1.3.0


In [2]:
import os
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

from ale_py import ALEInterface

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Load Environment


In [3]:
environment_name = 'CartPole-v1'

env = gym.make(environment_name, render_mode="rgb_array")

In [4]:
episodes = 5

for episode in range(1, episodes + 1):
    obs, info = env.reset()
    done = False
    score = 0
    
    while not done:
        env.render()
        action = env.action_space.sample()
        obs, reward, finished, truncated, info = env.step(action)
        
        score += reward
        done = finished or truncated
    
    print("Episode: {} Score: {}".format(episode, score))

env.close()

Episode: 1 Score: 11.0
Episode: 2 Score: 20.0
Episode: 3 Score: 22.0
Episode: 4 Score: 34.0
Episode: 5 Score: 13.0


In [5]:
env.action_space

Discrete(2)

In [ ]:
env.action_space.sample()

np.int64(1)

In [7]:
env.observation_space

Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)

In [8]:
env.observation_space.sample()

array([-1.5638912 , -0.636775  ,  0.06309152, -0.6183278 ], dtype=float32)

### Train RL model


In [9]:
log_path = os.path.join("Training", "Logs")
os.path.join("Training", "Saved Models")

log_path

'Training/Logs'

In [10]:
env = gym.make(environment_name)
env = DummyVecEnv([lambda: env])

model = PPO("MlpPolicy", env, verbose=1, tensorboard_log=log_path)

Using cpu device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [11]:
model.learn(total_timesteps=10000)

Logging to Training/Logs/PPO_1
-----------------------------
| time/              |      |
|    fps             | 1171 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
----------------------------------------
| time/                   |            |
|    fps                  | 852        |
|    iterations           | 2          |
|    time_elapsed         | 4          |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.00800141 |
|    clip_fraction        | 0.0936     |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.686     |
|    explained_variance   | -0.00615   |
|    learning_rate        | 0.0003     |
|    loss                 | 7.28       |
|    n_updates            | 10         |
|    policy_gradient_loss | -0.014     |
|    value_loss           | 50.1       |
----------------------------------------
---------------------

### Save and reload model


In [12]:
PPO_Path = os.path.join("Training", "Saved Models", "PPO_MODEL_CartPole")

In [13]:
model.save(PPO_Path)

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/save_util.py:284: UserWarning: Path 'Training/Saved Models' does not exist. Will create it.
  warnings.warn(f"Path '{path.parent}' does not exist. Will create it.")


In [14]:
del model

In [15]:
model = PPO.load(PPO_Path, env=env)

### Evaluation


In [16]:
evaluate_policy(model, env, n_eval_episodes=10, render=True)

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/base_vec_env.py:259: UserWarning: You tried to call render() but no `render_mode` was passed to the env constructor.
  warnings.warn("You tried to call render() but no `render_mode` was passed to the env constructor.")


(np.float64(315.9), np.float64(125.44038424686046))